# Plot Eiger2 stats vs. a motor

This worked example shows how to use the Eiger2 area-detector's
**Stats1** plugin to record total counts and maximum pixel value during
a Bluesky scan, then re-plot the result from the catalog.

It assumes:

- The notebook is being run inside the beamline's `3idc-bits` conda
  environment.  On the beamline workstation, activate it first:
  `conda activate 3idc-bits`.  Anywhere else, this example will not
  work -- `id3c`, the Tiled `cat`, and the Eiger2 PVs are only
  available there.
- The `eiger2` and `sim_motor` devices are configured in
  `devices.yml` (they ship with the instrument).
- You can reach the EPICS PVs for the Eiger2 from this host.  On a
  workstation that cannot, every code cell will run as far as the
  first CA put, then fail with `DisconnectedError` -- expected and
  harmless for reading along.


## 1. Start the session

From a terminal on the beamline workstation:

```bash
conda activate 3idc-bits
jupyter lab          # or 'ipython' for the command-line session
```

You can also run this notebook from a VS Code session with the
Python and Jupyter extensions installed, as long as VS Code's
kernel picker is pointed at the `3idc-bits` conda environment.

Then in the first cell:


In [ ]:
from id3c.startup import *
print(eiger2.name, sim_motor.name)


eiger2 sim_motor


## 2. Make the stats signals visible to the RunEngine

Every ophyd Signal (and Device) has a `kind` attribute that decides
which of `read()` / `read_configuration()` returns it and whether the
BestEffortCallback plots it automatically.  See
[EPICS -> ophyd](../tutorials/epics_to_ophyd.md#the-kind-attribute)
for the full story.

By default the Eiger2's Stats1 plugin and its child signals are
`kind="omitted"`.  Three changes:

1. Set `stats1.kind` to `"normal"` so the plugin itself is visited
   when the parent device is read.
2. Set `stats1.total.kind` and `stats1.max_value.kind` to `"hinted"`
   so they appear in `stats1.read()` and get plotted by `bec`.

If you set only the children to `"hinted"` without also fixing the
plugin, the children will not be visited.


In [ ]:
eiger2.stats1.kind = "normal"
eiger2.stats1.total.kind = "hinted"
eiger2.stats1.max_value.kind = "hinted"
eiger2.stats1.read()


{'eiger2_stats1_total': {'value': 12345678.0, 'timestamp': 1719000000.0},
 'eiger2_stats1_max_value': {'value': 1024.0, 'timestamp': 1719000000.0}}


## 3. Count 5 times to verify the stats stream

`bp.count` triggers the detector and reads its hinted signals on each
iteration.  Because we passed the two stats components (not the whole
`eiger2` device), no image data is recorded -- just the two stats.


In [ ]:
count_uid, = RE(
    bp.count([eiger2.stats1.total, eiger2.stats1.max_value], num=5)
)
count_uid




Transient Scan ID: 1     Time: 2026-05-29 14:00:00
Persistent Unique Scan ID: 'b90b5bf4-bcf0-4f89-b92d-734cab21dbf2'
New stream: 'primary'
+-----------+------------+------------------------+-------------------------+
|   seq_num |       time | eiger2_stats1_total    | eiger2_stats1_max_value |
+-----------+------------+------------------------+-------------------------+
|         1 | 14:00:00.1 |               12345678 |                    1024 |
|         2 | 14:00:00.2 |               12345710 |                    1024 |
|         3 | 14:00:00.3 |               12345692 |                    1024 |
|         4 | 14:00:00.4 |               12345703 |                    1024 |
|         5 | 14:00:00.5 |               12345699 |                    1024 |
+-----------+------------+------------------------+-------------------------+
generator count ['b90b5bf4'] (scan num: 1)


'b90b5bf4-bcf0-4f89-b92d-734cab21dbf2'

## 4. Inspect the run

Every `RE(...)` invocation returns a sequence of run UIDs (one per
run, usually length 1).  Above we wrote `count_uid, = RE(...)`:
the **trailing comma is significant** -- it is Python's
iterable-unpacking syntax, asking for the single element of a
one-item sequence.  Without the comma you would bind `count_uid`
to the whole sequence and `cat[count_uid]` below would fail.

Capturing the UID is more robust than `cat[-1]`, which can refer
to a different run if another session writes to the catalog in
between or if you `RE(...)` again in the same cell.

`run.primary.read()` returns the main event stream as an
[xarray.Dataset](https://docs.xarray.dev/).


In [ ]:
run = cat[count_uid]
ds = run.primary.read()
ds


<xarray.Dataset>
Dimensions:                   (time: 5)
Coordinates:
  * time                     (time) float64 1.719e+09 1.719e+09 ...
Data variables:
    eiger2_stats1_total      (time) float64 1.234e+07 1.234e+07 ...
    eiger2_stats1_max_value  (time) float64 1024.0 1024.0 ...


## 5. Scan `eiger2` against `sim_motor`

This time we pass the whole `eiger2` device (not just the stats
components), so the detector's image data is also recorded.  We scan
`sim_motor` from -1 to +1 in 3 steps to keep the example short.

The `bec` BestEffortCallback opens a live plot as the scan runs and
prints a table when it finishes.


In [ ]:
scan_uid, = RE(bp.scan([eiger2], sim_motor, -1, 1, 3))
scan_uid




Transient Scan ID: 2     Time: 2026-05-29 14:01:00
Persistent Unique Scan ID: 'cd57a32b-23e8-4856-aa89-032cf928c3c7'
New stream: 'primary'
+-----------+------------+------------+-------------------------+----------------------+
|   seq_num |       time |  sim_motor | eiger2_stats1_max_value | eiger2_stats1_total  |
+-----------+------------+------------+-------------------------+----------------------+
|         1 | 14:01:00.5 |    -1.0000 |                    1019 |             12331284 |
|         2 | 14:01:01.5 |     0.0000 |                    1024 |             12345678 |
|         3 | 14:01:02.5 |     1.0000 |                    1023 |             12343007 |
+-----------+------------+------------+-------------------------+----------------------+
generator scan ['cd57a32b'] (scan num: 2)


'cd57a32b-23e8-4856-aa89-032cf928c3c7'

## 6. Re-plot from the catalog

BEC already plotted the scan live.  To re-plot from the catalog
after the fact -- e.g. on a different day, or with peak-statistics
overlaid -- use `apstools.utils.plotxy`.  It accepts a run (or a
reference like `-1` or a UID) and the names of the x and y signals,
and returns a dict of computed statistics (centroid, FWHM, etc.).


In [ ]:
from apstools.utils import plotxy

stats = plotxy(cat[scan_uid], "sim_motor", "eiger2_stats1_total")
stats


{2: {'min_x': -1.0,
  'max_x': 1.0,
  'mean_x': 0.0,
  'min_y': 12331284.0,
  'max_y': 12345678.0,
  'mean_y': 12339989.67,
  'centroid': 0.00039,
  'sigma': 0.81649,
  'fwhm': 1.92256}}


The figure opens in a separate matplotlib window (or inline, if
you're running this notebook in JupyterLab).  Centroid and FWHM are
drawn as gray reference lines over the curve.


## See also

- [How to run a scan](run_a_scan.md) -- the full menu of Bluesky
  scan plans.
- [How to inspect data](inspect_data.md) -- reading runs back from
  the Tiled catalog.
- [How to visualize HDF5 image files](visualize_hdf5.md) -- area-
  detector image data (the current state at 3-ID-C is documented).
- [EPICS -> ophyd](../tutorials/epics_to_ophyd.md) -- the `kind`
  attribute and the device model.
- [`apstools.utils.plotxy` source](https://github.com/BCDA-APS/apstools/blob/main/apstools/utils/plot.py).
